[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/14_indexacion_ann/14_proyecto_indexacion_ann.ipynb)

# Proyecto 14 · Índice de Políglota: curvas recall@10 vs latencia vs memoria

**Nivel:** 🟠 Avanzado · **Duración:** ≈ 2 h · **GPU:** no necesaria (CPU) · **Cómputo:** 0 unidades de Colab.

**Conexión.** *Viene de:* módulo 12 (el embedder y la dimensión elegidos) y 13 (el retriever denso del sistema híbrido). *Va hacia:* módulo 15 (evaluar el sistema completo) y 18 (operar el índice: Qdrant/pgvector, versionado, migración).

## 1 · Contexto de negocio
Políglota Labs indexará **10 millones de fragmentos**. El CTO te entrega la «ficha técnica» que debe cumplir el índice en producción:

| Requisito | Valor |
|---|---|
| Recall@10 contra búsqueda exacta | ≥ 0.90 |
| Latencia p99 (batch = 1, un hilo) | ≤ 50 % del p99 de la fuerza bruta medida en tu máquina (al menos 2× más rápido) |
| Memoria del índice (extrapolada a 10 M) | ≤ 3 × la de la fuerza bruta `float32` |

Tu trabajo: medir con el protocolo correcto varias familias de índices FAISS sobre **vectores de Políglota**, dibujar las curvas, extraer la frontera de Pareto y **recomendar** una configuración (o decir con datos por qué ninguna cumple).

## 2 · Dataset
Embeddings del corpus Políglota (`carga_corpus`, módulo 00; sintético con `⚠️` si no hay acceso) calculados con el embedder del módulo 12 (si no se puede descargar, el fallback TF-IDF+SVD de 64 dimensiones). Como el corpus tiene solo ~400 documentos, se **amplía con copias ruidosas** (vecinos difíciles) hasta `N` vectores; las queries son las del corpus. **Con vectores reales de un modelo los resultados serán distintos de los del fallback**: lo importante es el procedimiento.

## 3 · Entregables
1. Funciones `exacto_topk`, `mide_latencia` y `tam_indice_MB` (protocolo de medición).
2. `construye` y `fija_parametro` para 4 familias FAISS: `Flat`, `IVF-Flat`, `IVF-PQ+refine` y `HNSW`.
3. `barrido`: tabla con recall@10, p50, p99, QPS, MB y MB extrapolados a 10 M para cada (familia, parámetro).
4. `no_dominados` (frontera de Pareto en 3 objetivos) y `elige_config` (mínima memoria que cumple el SLA).
5. Dos gráficas: recall vs latencia (con memoria en la leyenda) y recall vs memoria; y una conclusión escrita.

## 4 · Rúbrica (umbrales calibrados con lo que MIDE la solución de referencia)
La solución de referencia se ejecutó en un entorno **sin acceso a Hugging Face** (corpus sintético y embedder de respaldo TF-IDF+SVD de 64 dimensiones), con `N` = 20 000 y un hilo; **no se verificó con un embedder real**. Los umbrales son invariantes relativas; el valor medido está en la última columna.

| # | Criterio | Umbral | Medido por la solución |
|---|---|---|---|
| R1 | La fila `Flat` tiene recall = 1.0 (el oráculo es exacto) | = 1.0 | 1.000 |
| R2 | p99 ≥ p50 en todas las filas | 100 % | 100 % de las filas |
| R3 | Memoria: `Flat` ≈ 4·N·D bytes (±5 %) y `HNSW` > `Flat` | cumplido | Flat 5.12 MB (teórico 5.12) · HNSW 8.00 MB |
| R4 | Dentro de cada familia el recall no baja al subir el parámetro de búsqueda (tolerancia 0.01) | ≥ 90 % de pares | 100 % de pares |
| R5 | La frontera de Pareto tiene ≥ 3 puntos y ≥ 2 familias | cumplido | 10 puntos, 4 familias |
| R6 | `elige_config` devuelve una fila que cumple el SLA (o `None` si ninguna lo cumple) | cumplido | IVF-Flat nprobe=4 (recall 0.95, p99 0.06 ms, 5.3 MB) |
| R7 | Conclusión escrita con ≥ 3 números de tus tablas | — | revisión manual |

## 5 · Preparación (celdas dadas)

In [ ]:
!pip install -q faiss-cpu numpy scipy scikit-learn matplotlib pandas sentence-transformers

In [ ]:
import os, sys, time, math, random, json, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")

import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)
FAST_DEV_RUN = True      # True: tamaños diminutos (CPU en minutos). Pon False en Colab con GPU para la escala completa.
N_THREADS = int(os.environ.get("N_THREADS", "1" if FAST_DEV_RUN else "4"))
os.environ.setdefault("OMP_NUM_THREADS", str(N_THREADS))

try:
    import torch
    torch.manual_seed(SEED)
    torch.set_num_threads(N_THREADS)
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:      # los fragmentos "desde cero" de NumPy funcionan sin torch
    torch, device = None, "cpu"
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · threads={N_THREADS}")

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out

In [ ]:
# Catálogo de embedders multilingües "pequeños" (caben en CPU/T4). Prefijos copiados de las tarjetas de modelo:
# VERIFÍCALOS en la tarjeta antes de usar un modelo en producción (cambian entre versiones).
MODELOS = {
    "multilingual-e5-small": dict(hf="intfloat/multilingual-e5-small", q="query: ", d="passage: "),
    "multilingual-e5-base": dict(hf="intfloat/multilingual-e5-base", q="query: ", d="passage: "),
    "paraphrase-multilingual-MiniLM-L12-v2": dict(hf="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", q="", d=""),
    "gte-multilingual-base": dict(hf="Alibaba-NLP/gte-multilingual-base", q="", d="", trust=True),
    "bge-m3": dict(hf="BAAI/bge-m3", q="", d=""),
    "nomic-embed-text-v2-moe": dict(hf="nomic-ai/nomic-embed-text-v2-moe", q="search_query: ", d="search_document: ", trust=True),
    "Qwen3-Embedding-0.6B": dict(hf="Qwen/Qwen3-Embedding-0.6B", d="",
                                 q="Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"),
    "embeddinggemma-300m": dict(hf="google/embeddinggemma-300m", q="task: search result | query: ", d="title: none | text: ", gated=True),
}

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

class HashSVDEmbedder:
    """Fallback SIN descargas: TF-IDF de n-gramas de caracteres + SVD. NO es multilingüe ni semántico:
    sirve para ejercitar el pipeline (truncado, cuantización, métricas) cuando HF no es accesible."""
    def __init__(self, dim: int = 64, ngram=(3, 5), analyzer: str = "char_wb"):
        self.dim = dim
        self.tfidf = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram, sublinear_tf=True, min_df=1)
        self.svd = TruncatedSVD(n_components=dim, random_state=SEED)
    def fit(self, textos):
        X = self.tfidf.fit_transform(textos)
        self.svd.n_components = min(self.dim, X.shape[1] - 1, X.shape[0] - 1)
        self.svd.fit(X)
        return self
    def encode(self, textos):
        Z = self.svd.transform(self.tfidf.transform(textos))
        return (Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-9)).astype(np.float32)

class Embedder:
    """Interfaz común: encode_queries / encode_docs (aplican el prefijo de cada modelo) -> float32 normalizado."""
    def __init__(self, clave, cfg, model=None, fallback=None):
        self.clave, self.cfg, self.model, self.fallback = clave, cfg, model, fallback
        self.is_real = model is not None
    def _enc(self, textos, prefijo, bs=16):
        if self.model is not None:
            E = self.model.encode([prefijo + t for t in textos], batch_size=bs, normalize_embeddings=True,
                                  convert_to_numpy=True, show_progress_bar=False)
            return E.astype(np.float32)
        return self.fallback.encode([prefijo + t for t in textos])
    def encode_queries(self, textos): return self._enc(textos, self.cfg.get("q", ""))
    def encode_docs(self, textos): return self._enc(textos, self.cfg.get("d", ""))
    @property
    def n_params(self):
        return sum(p.numel() for p in self.model.parameters()) if self.model is not None else 0

def cargar_embedder(clave, textos_ajuste=None, max_len=256):
    """Carga con sentence-transformers; si no se puede (sin red, modelo con licencia, falta de RAM) cae al fallback."""
    cfg = MODELOS[clave]
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer(cfg["hf"], device=device, trust_remote_code=cfg.get("trust", False))
        m.max_seq_length = min(m.max_seq_length or max_len, max_len)
        return Embedder(clave, cfg, model=m)
    except Exception as e:
        print(f"[embedder] {clave}: no se pudo cargar ({type(e).__name__}: {str(e)[:70]})")
        if textos_ajuste is None:
            return None
        print(f"[embedder] {clave}: usando fallback TF-IDF+SVD (NO representa al modelo real)")
        return Embedder(clave + " (fallback)", cfg, fallback=HashSVDEmbedder().fit(textos_ajuste))

In [ ]:
import faiss
faiss.omp_set_num_threads(N_THREADS)          # medición justa: un hilo
import pandas as pd
import matplotlib.pyplot as plt
def recall_vs_gt(idx, gt):
    """Neighbor recall@k: fracción del top-k exacto que devuelve el índice aproximado (ignora los -1 de FAISS)."""
    return float(np.mean([len(set(a[a >= 0].tolist()) & set(b.tolist())) / len(b) for a, b in zip(idx, gt)]))

## 6 · Paso 1 — Protocolo de medición (TODO 1–3)
- **TODO 1 `exacto_topk(Q, X, k)`**: fuerza bruta por producto interno (vectores normalizados) procesando las queries por bloques de 64; devuelve los **índices** del top-`k` ordenados de mayor a menor score. Es el oráculo para el recall.
- **TODO 2 `mide_latencia(fn, Q)`**: llama a `fn(q)` para **cada query por separado** (batch = 1), tras un calentamiento de 10 queries, y devuelve `(p50_ms, p99_ms, qps)` donde `qps = 1000 / media_ms`.
- **TODO 3 `tam_indice_MB(index)`**: tamaño real del índice serializado en MB (`faiss.serialize_index`).

<details><summary>🪜 Pista 1 (idea)</summary>
Para el top-k sin ordenar todo: <code>np.argpartition</code> y luego ordena solo esos k. Para latencia, cronometra con <code>time.perf_counter()</code> una llamada por query y usa <code>np.percentile</code>.
</details>
<details><summary>🪜 Pista 2 (casi la API)</summary>
<code>idx = np.argpartition(-S, k, axis=1)[:, :k]</code> · <code>orden = np.take_along_axis(S, idx, 1).argsort(1)[:, ::-1]</code> · <code>faiss.serialize_index(index).nbytes / 1e6</code>.
</details>

In [ ]:
def exacto_topk(Q, X, k=10):
    # TODO 1: top-k exacto (índices) por producto interno, en bloques de 64 queries
    raise NotImplementedError("TODO 1")

def mide_latencia(fn, Q, calentamiento=10):
    # TODO 2: devuelve (p50_ms, p99_ms, qps) llamando a fn(q) una query cada vez
    raise NotImplementedError("TODO 2")

def tam_indice_MB(index):
    # TODO 3: tamaño serializado en MB
    raise NotImplementedError("TODO 3")

try:
    _X = np.random.default_rng(0).normal(size=(500, 16)).astype("float32"); _X /= np.linalg.norm(_X, axis=1, keepdims=True)
    _gt = exacto_topk(_X[:5], _X, 3)
    print("el vecino más cercano de cada vector es él mismo:", bool((_gt[:, 0] == np.arange(5)).all()))
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## 7 · Paso 2 — Vectores de Políglota (dado)
Embebemos documentos y queries con el embedder del módulo 12 y ampliamos el corpus con copias ruidosas hasta `N` vectores (vecinos difíciles). Las queries y el índice quedan **separados** (las queries nunca se añaden al índice).

In [ ]:
CORPUS = carga_corpus(FAST_DEV_RUN)
DOC_TXT, Q_TXT = [d["text"] for d in CORPUS["docs"]], [q["text"] for q in CORPUS["queries"]]
emb = None
for clave in ("multilingual-e5-small", "paraphrase-multilingual-MiniLM-L12-v2"):
    emb = cargar_embedder(clave)
    if emb is not None:
        break
if emb is None:
    emb = cargar_embedder("multilingual-e5-small", textos_ajuste=DOC_TXT)
E_d, E_q = emb.encode_docs(DOC_TXT), emb.encode_queries(Q_TXT)

def aumenta(E, n_objetivo, sigma=0.8, seed=0):
    """Copias ruidosas de cada vector (normalizadas) hasta n_objetivo vectores: crea vecinos difíciles."""
    rng = np.random.default_rng(seed)
    bloques, total = [E], len(E)
    while total < n_objetivo:
        ruido = rng.normal(size=E.shape)
        ruido *= sigma / np.linalg.norm(ruido, axis=1, keepdims=True)
        X_ = E + ruido
        bloques.append((X_ / np.linalg.norm(X_, axis=1, keepdims=True)).astype(np.float32)); total += len(E)
    return np.vstack(bloques)[:n_objetivo]

N = 20000 if FAST_DEV_RUN else 200000
X = np.ascontiguousarray(aumenta(E_d, N))
Q = np.ascontiguousarray(E_q.astype(np.float32))
D = X.shape[1]
N_PROD = 10_000_000
print(f"embedder={emb.clave} (real={emb.is_real}) · X: {X.shape} · Q: {Q.shape}")

## 8 · Paso 3 — Construir y parametrizar 4 familias (TODO 4)
Implementa `construye(familia, X)` y `fija_parametro(ix, familia, valor)`:

| Familia | Construcción (producto interno) | Parámetro de búsqueda |
|---|---|---|
| `Flat` | `IndexFlatIP(D)` | — (usa `0`) |
| `IVF-Flat` | `IndexIVFFlat(IndexFlatIP(D), D, nlist)`, `nlist = int(sqrt(N))`, entrenar y añadir | `nprobe` |
| `IVF-PQ+refine` | `IndexIVFPQ(IndexFlatIP(D), D, nlist, m, 8)` envuelto en `IndexRefineFlat`; `m` = el mayor divisor de `D` en {32,16,8,4}; entrenar y añadir | `nprobe` (del índice base) con `k_factor = 10` |
| `HNSW` | `IndexHNSWFlat(D, 16, METRIC_INNER_PRODUCT)` con `efConstruction = 100` | `efSearch` |

<details><summary>🪜 Pista 1 (idea)</summary>
Los índices con entrenamiento (<code>is_trained == False</code>) necesitan <code>ix.train(X)</code> antes de <code>ix.add(X)</code>. Para <code>IndexRefineFlat</code> el <code>nprobe</code> se fija sobre el índice IVF interno.
</details>
<details><summary>🪜 Pista 2 (casi la API)</summary>
<code>base = faiss.IndexIVFPQ(faiss.IndexFlatIP(D), D, nlist, m, 8, faiss.METRIC_INNER_PRODUCT)</code> · <code>ix = faiss.IndexRefineFlat(base); KEEP.append(base)</code> (FAISS no es dueño de <code>base</code>: mantén la referencia) · <code>faiss.extract_index_ivf(ix.base_index).nprobe = v; ix.k_factor = 10</code> · HNSW: <code>ix.hnsw.efSearch = v</code>.
</details>

In [ ]:
KEEP = []                       # referencias que FAISS no posee (índice base de IndexRefineFlat)
FAMILIAS = {"Flat": [0], "IVF-Flat": [1, 2, 4, 8, 16, 32], "IVF-PQ+refine": [2, 4, 8, 16, 32], "HNSW": [8, 16, 32, 64, 128]}

def construye(familia, X):
    # TODO 4.a: devuelve el índice FAISS ya entrenado y con X añadido
    raise NotImplementedError("TODO 4.a")

def fija_parametro(ix, familia, valor):
    # TODO 4.b: fija nprobe / efSearch (y k_factor=10 en IVF-PQ+refine)
    raise NotImplementedError("TODO 4.b")

try:
    _ix = construye("IVF-Flat", X[:2000]); fija_parametro(_ix, "IVF-Flat", 4)
    print("IVF-Flat construido:", _ix.ntotal, "vectores")
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## 9 · Paso 4 — El barrido (TODO 5)
Para cada familia y parámetro: recall@10 contra `exacto_topk`, p50/p99/QPS con batch=1, `MB` real y `MB_10M` (= `MB * N_PROD / N`, extrapolación lineal: suficiente para comparar, **no** para dimensionar un clúster). Devuelve un `DataFrame` con columnas `familia, param, recall, p50_ms, p99_ms, QPS, MB, MB_10M, seg_construir`.

<details><summary>🪜 Pista 2 (casi la API)</summary>
Construye el índice una vez por familia (mide <code>seg_construir</code>), y para cada valor: <code>fija_parametro</code>, <code>_, I = ix.search(Q, 10)</code>, <code>mide_latencia(lambda q: ix.search(q[None], 10), Q)</code>.
</details>

In [ ]:
def barrido(X, Q, GT):
    # TODO 5: recorre FAMILIAS y devuelve el DataFrame descrito arriba
    raise NotImplementedError("TODO 5")

try:
    GT = exacto_topk(Q, X, 10)
    RES = barrido(X, Q, GT)
    display(RES.round(3).head(10))
except NotImplementedError as e:
    GT = RES = None
    print("⏳ pendiente:", e)

## 10 · Paso 5 — Frontera de Pareto y recomendación (TODO 6–7)
- `no_dominados(df, maximizar, minimizar)`: filas para las que **no existe otra** que sea al menos igual en todos los objetivos y estrictamente mejor en alguno. Úsalo con `maximizar=["recall"]`, `minimizar=["p99_ms", "MB"]`.
- `elige_config(df, recall_min, p99_max_ms, mem_max_MB10M)`: entre las filas que cumplen el SLA, la de **menor `MB`** (desempate: menor `p99_ms`); `None` si ninguna cumple.

In [ ]:
RECALL_MIN, P99_MAX_MS = 0.90, None          # P99_MAX_MS = 0.5 x p99 de la fuerza bruta: se fija tras el barrido
MEM_MAX_MB10M = 3 * 4 * D * N_PROD / 1e6          # 3 x la fuerza bruta float32 extrapolada a 10 M

def no_dominados(df, maximizar, minimizar):
    # TODO 6: devuelve el subconjunto de filas no dominadas
    raise NotImplementedError("TODO 6")

def elige_config(df, recall_min, p99_max_ms, mem_max_MB10M):
    # TODO 7: fila de menor memoria que cumple el SLA (None si ninguna)
    raise NotImplementedError("TODO 7")

def dibuja(df, frente):
    colores = {"Flat": "#6B7280", "IVF-Flat": "#1F6FEB", "IVF-PQ+refine": "#9467BD", "HNSW": "#F28E2B"}
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    for fam, g in df.groupby("familia"):
        axs[0].plot(g["p50_ms"], g["recall"], "o-", color=colores[fam], label=f"{fam} ({g['MB'].iloc[0]:.1f} MB)")
        axs[1].scatter(g["MB"], g["recall"], color=colores[fam], s=30, label=fam)
    axs[0].scatter(frente["p50_ms"], frente["recall"], s=90, facecolors="none", edgecolors="k", label="no dominados")
    axs[0].axhline(RECALL_MIN, color="k", ls=":"); axs[0].set_xscale("log"); axs[0].set_xlabel("p50 (ms, log)"); axs[0].set_ylabel("recall@10"); axs[0].legend(fontsize=7); axs[0].grid(alpha=.3, which="both")
    axs[1].set_xscale("log"); axs[1].set_xlabel("memoria del índice (MB, log)"); axs[1].set_ylabel("recall@10"); axs[1].grid(alpha=.3, which="both")
    plt.tight_layout(); plt.show()

try:
    P99_MAX_MS = 0.5 * float(RES.loc[RES["familia"] == "Flat", "p99_ms"].iloc[0])
    FRENTE = no_dominados(RES, ["recall"], ["p99_ms", "MB"])
    ELEGIDA = elige_config(RES, RECALL_MIN, P99_MAX_MS, MEM_MAX_MB10M)
    dibuja(RES, FRENTE)
    print("Configuración elegida:", None if ELEGIDA is None else ELEGIDA[["familia", "param", "recall", "p99_ms", "MB", "MB_10M"]].to_dict())
except (NotImplementedError, TypeError, AttributeError) as e:
    FRENTE = ELEGIDA = None
    print("⏳ pendiente:", repr(e))

## 11 · Autocomprobación de la rúbrica

In [ ]:
def autograder():
    ch = {}
    try:
        flat = RES[RES["familia"] == "Flat"]
        ch["R1 Flat tiene recall = 1.0"] = bool((flat["recall"] >= 0.999).all() and len(flat))
        ch["R2 p99 ≥ p50"] = bool((RES["p99_ms"] >= RES["p50_ms"] - 1e-9).all())
        mb = RES.groupby("familia")["MB"].first()
        ch["R3 memoria coherente"] = bool(abs(mb["Flat"] - 4 * N * D / 1e6) / (4 * N * D / 1e6) < 0.05 and mb["HNSW"] > mb["Flat"])
        pares = tot = 0
        for fam, g in RES.groupby("familia"):
            r = g.sort_values("param")["recall"].to_numpy()
            pares += len(r) - 1; tot += int((np.diff(r) >= -0.01).sum())
        ch["R4 recall monótono en el parámetro (≥ 90 % de pares)"] = bool(pares == 0 or tot / pares >= 0.90)
    except Exception as e:
        for k in ("R1 Flat tiene recall = 1.0", "R2 p99 ≥ p50", "R3 memoria coherente", "R4 recall monótono en el parámetro (≥ 90 % de pares)"):
            ch.setdefault(k, False)
    try:
        ch["R5 Pareto ≥ 3 puntos y ≥ 2 familias"] = bool(len(FRENTE) >= 3 and FRENTE["familia"].nunique() >= 2)
    except Exception:
        ch["R5 Pareto ≥ 3 puntos y ≥ 2 familias"] = False
    try:
        ok = RES[(RES["recall"] >= RECALL_MIN) & (RES["p99_ms"] <= P99_MAX_MS) & (RES["MB_10M"] <= MEM_MAX_MB10M)]
        ch["R6 elige_config respeta el SLA"] = bool((len(ok) == 0 and ELEGIDA is None) or (ELEGIDA is not None and ELEGIDA["recall"] >= RECALL_MIN
                                                   and ELEGIDA["p99_ms"] <= P99_MAX_MS and ELEGIDA["MB_10M"] <= MEM_MAX_MB10M and ELEGIDA["MB"] <= ok["MB"].min() + 1e-9))
    except Exception:
        ch["R6 elige_config respeta el SLA"] = False
    for k, v in ch.items():
        print("✅" if v else "❌", k)
    return ch

_ = autograder()

## 12 · Tu conclusión (R7)
En ≤ 10 líneas, con números de tus tablas: ¿qué configuración recomiendas para 10 M de fragmentos y por qué? ¿Qué familia gana en latencia, cuál en memoria y cuál en tiempo de construcción? ¿Qué cambiaría si el SLA exigiera recall ≥ 0.99? ¿Qué riesgos ves al extrapolar de `N` = 20 000 a 10 M?

> *(Edita esta celda.)*

## 13 · Retos extra (nivel experto)
1. **Matryoshka + índice** (módulo 12): trunca el embedder a D/2 y D/4 y repite el barrido; ¿qué combinación (dimensión × familia) domina?
2. **Mini-HNSW de la lección** (sección 5) frente a `IndexHNSWFlat` con tus vectores: ¿cuántas distancias evalúa cada una para el mismo recall?
3. **Búsqueda filtrada**: añade un atributo `lang` (`es`/`en`/`code`) a cada vector (el corpus lo trae) y mide recall/latencia de pre-filtro, post-filtro y filtro en grafo (lección, sección 9).
4. **Binaria + rescoring con FAISS** (`IndexBinaryFlat` / `IndexBinaryIVF`): compara con `IVF-PQ+refine` en memoria y recall.
5. **Curva con N creciente**: repite con N = 20 k, 50 k, 100 k y grafica cómo escala la latencia de cada familia.

## 14 · Reflexión (producción e hilo conductor)
- ¿A partir de qué `N` empieza a ganar un ANN a la fuerza bruta en tu máquina? (reto 5)
- ¿Qué parte de tu elección cambia si el 30 % de las consultas llevan un filtro por idioma o versión?
- ¿Cómo detectarías en producción que el recall del índice se ha degradado (deriva, borrados acumulados)? (módulo 18)
- Si migras de embedder, ¿qué hay que reconstruir y cómo evitas el *downtime*?
- ¿Cuánto cuesta (RAM, $/mes) el índice elegido a 10 M y a 100 M de fragmentos?

---
# ⛔ SPOILER — intenta resolverlo primero
Las celdas siguientes **sobrescriben** tus funciones con la solución de referencia.

In [ ]:
def exacto_topk(Q, X, k=10):
    """Fuerza bruta: el 'oráculo' para medir recall. Q (nq,d), X (n,d) normalizados -> índices (nq,k)."""
    out = np.empty((len(Q), k), dtype=np.int64)
    for s in range(0, len(Q), 64):
        S = Q[s:s + 64] @ X.T
        idx = np.argpartition(-S, k, axis=1)[:, :k]
        out[s:s + 64] = np.take_along_axis(idx, np.take_along_axis(S, idx, 1).argsort(1)[:, ::-1], 1)
    return out

def mide_latencia(fn, Q, calentamiento=10, repeticiones=1):
    """Latencia por query con batch=1 (así se sirve en producción). fn(q: (d,)) -> cualquier cosa. Devuelve p50, p99 (ms) y QPS."""
    for q in Q[:calentamiento]:
        fn(q)
    t = []
    for _ in range(repeticiones):
        for q in Q:
            t0 = time.perf_counter(); fn(q); t.append(time.perf_counter() - t0)
    t = np.array(t) * 1e3
    return float(np.percentile(t, 50)), float(np.percentile(t, 99)), float(1e3 / t.mean())

def tam_indice_MB(index):
    """Tamaño real serializado del índice FAISS en MB (incluye vectores, listas y grafo)."""
    return faiss.serialize_index(index).nbytes / 1e6

In [ ]:
def construye(familia, X):
    nlist = int(np.sqrt(len(X)))
    if familia == "Flat":
        ix = faiss.IndexFlatIP(D)
    elif familia == "IVF-Flat":
        ix = faiss.IndexIVFFlat(faiss.IndexFlatIP(D), D, nlist, faiss.METRIC_INNER_PRODUCT)
    elif familia == "IVF-PQ+refine":
        m = max(d_ for d_ in (32, 16, 8, 4) if D % d_ == 0)
        base = faiss.IndexIVFPQ(faiss.IndexFlatIP(D), D, nlist, m, 8, faiss.METRIC_INNER_PRODUCT)
        KEEP.append(base)
        ix = faiss.IndexRefineFlat(base)
    elif familia == "HNSW":
        ix = faiss.IndexHNSWFlat(D, 16, faiss.METRIC_INNER_PRODUCT); ix.hnsw.efConstruction = 100
    else:
        raise ValueError(familia)
    if not ix.is_trained:
        ix.train(X)
    ix.add(X)
    return ix

def fija_parametro(ix, familia, valor):
    if familia == "IVF-Flat":
        ix.nprobe = valor
    elif familia == "IVF-PQ+refine":
        faiss.extract_index_ivf(ix.base_index).nprobe = valor
        ix.k_factor = 10
    elif familia == "HNSW":
        ix.hnsw.efSearch = valor

def barrido(X, Q, GT):
    filas = []
    for familia, valores in FAMILIAS.items():
        t0 = time.perf_counter(); ix = construye(familia, X); t_b = time.perf_counter() - t0
        mb = tam_indice_MB(ix)
        for v in valores:
            fija_parametro(ix, familia, v)
            _, I = ix.search(Q, 10)
            p50, p99, qps = mide_latencia(lambda q: ix.search(q[None], 10), Q)
            filas.append(dict(familia=familia, param=v, recall=recall_vs_gt(I, GT), p50_ms=p50, p99_ms=p99, QPS=qps, MB=mb,
                              MB_10M=mb * N_PROD / len(X), seg_construir=t_b))
    return pd.DataFrame(filas)

def no_dominados(df, maximizar, minimizar):
    A = df[maximizar].to_numpy() * 1.0
    B = df[minimizar].to_numpy() * 1.0
    M = np.hstack([A, -B])                                     # todos "mayor es mejor"
    keep = []
    for i in range(len(M)):
        domina = ((M >= M[i]).all(1)) & ((M > M[i]).any(1))
        keep.append(not domina.any())
    return df[np.array(keep)].sort_values("MB")

def elige_config(df, recall_min, p99_max_ms, mem_max_MB10M):
    ok = df[(df["recall"] >= recall_min) & (df["p99_ms"] <= p99_max_ms) & (df["MB_10M"] <= mem_max_MB10M)]
    return None if ok.empty else ok.sort_values(["MB", "p99_ms"]).iloc[0]

GT = exacto_topk(Q, X, 10)
RES = barrido(X, Q, GT)
display(RES.round(3))
P99_MAX_MS = 0.5 * float(RES.loc[RES["familia"] == "Flat", "p99_ms"].iloc[0])
FRENTE = no_dominados(RES, ["recall"], ["p99_ms", "MB"])
ELEGIDA = elige_config(RES, RECALL_MIN, P99_MAX_MS, MEM_MAX_MB10M)
dibuja(RES, FRENTE); plt.show()
print("Configuración elegida:", None if ELEGIDA is None else ELEGIDA[["familia", "param", "recall", "p99_ms", "MB", "MB_10M"]].to_dict())
_ = autograder()

### Conclusión de referencia (ejemplo)
Con los números de la tabla anterior: la fuerza bruta da recall 1.0 como oráculo; IVF-Flat y HNSW alcanzan el recall objetivo con el parámetro de búsqueda más bajo que lo cumple, IVF-PQ+refine ocupa **más** que IVF-Flat porque, además de los códigos PQ, guarda los vectores completos para el re-ranking (en producción esos vectores irían a disco, no a RAM), y HNSW es el más grande y el más lento de construir. La configuración recomendada es la de **menor memoria que cumple recall, p99 y límite de memoria** (ver `ELEGIDA`); si ninguna cumple, los datos justifican relajar el SLA o cambiar de familia. Riesgos: extrapolar linealmente de 20 000 a 10 M, que el recall depende del embedder y de la dimensión, y que el p99 de una máquina de pruebas no es el de producción.